# Brazil IS-LM Model

In [16]:
# Importing necessary packages
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.widgets import Slider, Button
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings("ignore")

try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    try:
        get_ipython().run_line_magic('matplotlib', 'notebook')
        print("✓ backend: notebook")
    except Exception:
        get_ipython().run_line_magic('matplotlib', 'inline')

In [17]:
# Stating model parameters & Equations
# 2024 as Brazil baseline 
BL = dict(
    # Fiscal / real-side
    G   = 19.5,    # government spending  (% GDP)
    t   = 0.33,    # tax rate
    NX  = 2.4,     # net exports          (% GDP)
    c   = 0.72,    # marginal propensity to consume (MPC)
    b   = 1.4,     # investment sensitivity to interest rate
    # Autonomous
    a   = 35.0,    # autonomous consumption
    I0  = 18.0,    # autonomous investment
    # Money market
    k   = 0.80,    # income elasticity of money demand
    h   = 2.10,    # interest elasticity of money demand
    MP  = 100.0,   # real money supply index  (M/P)
    # Inflation
    pi  = 4.83,    # expected inflation  (IPCA 2024)
)

Y_RANGE = np.linspace(50, 150, 700)   # output grid for plotting
VIS     = (-6.0, 23.0)                # y-axis limits

# Core model functions 
def alpha(p):
    """Keynesian leakage:  α = 1 − c(1−t)"""
    return 1.0 - p['c'] * (1.0 - p['t'])

def A_val(p):
    """Autonomous expenditure:  A = a + I₀ + G + NX"""
    return p['a'] + p['I0'] + p['G'] + p['NX']

def IS(Y, p):
    """IS curve:  r = [A − Y·α] / b"""
    return (A_val(p) - Y * alpha(p)) / p['b']

def LM(Y, p):
    """LM curve:  r = (k·Y − M/P) / h"""
    return (p['k'] * Y - p['MP']) / p['h']

def equilibrium(p):
    """Solve Y* and r* analytically."""
    al = alpha(p);   A = A_val(p)
    Y  = (p['h'] * A  + p['b'] * p['MP']) / (p['h'] * al + p['b'] * p['k'])
    r  = (p['k'] * Y  - p['MP']) / p['h']
    return float(Y), float(r)

def multiplier(p):
    """Fiscal multiplier  μ = 1/α"""
    return 1.0 / alpha(p)

def nominal_rate(p, r):
    """Fisher equation:  i* = r* + πᵉ"""
    return r + p['pi']

def clip_curve(Y, r, vis=VIS):
    """Keep only the visible portion of a curve."""
    m = (r >= vis[0]) & (r <= vis[1])
    return Y[m], r[m]

def copy_params(p, **overrides):
    """Return a modified copy of a parameter dict."""
    q = dict(p);  q.update(overrides);  return q

#  Baseline equilibrium 
BL_Y, BL_r = equilibrium(BL)
BL_i        = nominal_rate(BL, BL_r)
BL_mu       = multiplier(BL)

print(f"✓ Baseline: Y*={BL_Y:.2f}  r*={BL_r:.2f}%  "
      f"i*={BL_i:.2f}%  μ={BL_mu:.4f}×")

✓ Baseline: Y*=134.71  r*=3.70%  i*=8.53%  μ=1.9320×


In [18]:
#Colour Palette
PAL = dict(
    IS      = "#1D9E75",   # teal-green  — IS baseline
    LM      = "#378ADD",   # blue        — LM baseline
    IS2     = "#E8593C",   # coral-red   — IS shifted
    LM2     = "#EF9F27",   # amber       — LM shifted
    EQ_BL   = "#534AB7",   # purple      — baseline equilibrium dot
    EQ_NEW  = "#E8593C",   # coral-red   — new equilibrium diamond
    EXP     = "#0F6E56",   # dark teal   — expansionary label colour
    CON     = "#993C1D",   # dark coral  — contractionary label colour
    MIX     = "#185FA5",   # dark blue   — mixed label colour
    GRID    = "#E0DDD6",
    BG      = "#FAFAF8",
    PANEL   = "#F4F2EC",
)

In [19]:
# Policy Stimulation Defination
SIMULATIONS = [
# MONETARY POLICY 
    dict(
        label     = "Money supply increase  (M/P ↑ +25)",
        kind      = "exp",
        channel   = "LM shifts RIGHT",
        delta     = dict(MP=+25),
        narrative = (
            "BCB open-market purchase → M/P rises → excess liquidity → "
            "bond yields fall → LM shifts right → r↓, Y↑."
        ),
        steps = [
            "BCB buys government bonds → money supply M rises",
            "Excess liquidity → households buy bonds → prices ↑, yields ↓",
            "LM shifts RIGHT  (for every Y, equilibrium r is now lower by ΔM·P/h)",
            "Lower r* → investment  I = I₀ − b·r  rises  (move along IS)",
            "New equilibrium:  Y* ↑   r* ↓   i* ↓  (Fisher: i = r + πᵉ falls too)",
        ],
    ),
    dict(
        label     = "Money supply contraction  (M/P ↓ −25)",
        kind      = "con",
        channel   = "LM shifts LEFT",
        delta     = dict(MP=-25),
        narrative = (
            "BCB tightens reserves / sells bonds → M/P falls → "
            "liquidity shortage → rates rise → LM shifts left → r↑, Y↓."
        ),
         steps = [
            "BCB sells bonds or raises reserve requirements → M falls",
            "Money shortage → excess demand for liquidity → interest rates ↑",
            "LM shifts LEFT  (for every Y, equilibrium r is now higher)",
            "Higher r* → private investment I falls  (move along IS, leftward)",
            "New equilibrium:  Y* ↓   r* ↑   i* ↑  — disinflationary intent",
        ],
    ),
    dict(
        label     = "Selic rate cut  (−300 bp)",
        kind      = "exp",
        channel   = "LM right + Fisher effect",
        delta     = dict(MP=+15, pi=-3),
        narrative = (
            "BCB cuts Selic target → overnight rates fall → credit eases → "
            "M/P effectively rises → LM shifts right. Fisher: lower i = r + πᵉ."
        ),
        steps = [
            "BCB signals Selic cut → overnight market rate falls across yield curve",
            "Lower nominal i → via Fisher (i = r + πᵉ) real rate r* falls",
            "Credit conditions ease → LM shifts rightward",
            "Investment & consumption rise → Y* increases along IS",
            "Currency may depreciate → NX improves → IS shifts right too (secondary)",
        ],
    ),
    dict(
        label     = "Selic rate hike  (+300 bp)",
        kind      = "con",
        channel   = "LM left + Fisher effect",
        delta     = dict(MP=-20, pi=+3),
        narrative = (
            "BCB raises Selic to fight inflation → money market tightens → "
            "real rate rises (Fisher) → LM shifts left → investment falls."
        ),
        steps = [
            "BCB raises Selic → overnight and term rates rise sharply",
            "Higher nominal i → real rate r* rises  (Fisher inversion)",
            "LM shifts LEFT; credit becomes expensive → private investment falls",
            "IS unchanged initially → new equilibrium: lower Y*, higher r*",
            "Intended:  cool aggregate demand, reduce inflationary pressure",
        ],
    ),
#  FISCAL POLICY 
    dict(
        label     = "Gov. spending increase  (G ↑ +3 pp GDP)",
        kind      = "exp",
        channel   = "IS shifts RIGHT",
        delta     = dict(G=+3),
        narrative = (
            "Treasury raises G → autonomous expenditure A rises → "
            "IS shifts right by μ·ΔG → higher Y* but r* also rises (crowding out)."
        ),
        steps = [
            "Government raises spending G → A = a + I₀ + G + NX increases directly",
            "IS shifts RIGHT by  μ × ΔG  (μ = 1/α is the fiscal multiplier)",
            "Higher Y → higher money demand L(r,Y) → r* rises  (move along LM)",
            "Higher r* → private investment I falls  (crowding-out effect)",
            "Net  ΔY < μ·ΔG  because crowding out partially offsets the stimulus",
        ],
    ),
    dict(
        label     = "Fiscal austerity  (G ↓ −3 pp GDP)",
        kind      = "con",
        channel   = "IS shifts LEFT",
        delta     = dict(G=-3),
        narrative = (
            "Government cuts spending → IS shifts left → Y* falls, "
            "r* also falls (crowding-in partially offsets). "
            "Effective if debt sustainability is the goal."
        ),
        steps = [
            "Government cuts spending G → aggregate demand A falls",
            "IS shifts LEFT by  μ × ΔG",
            "Lower Y → lower money demand → r* falls  (move along LM)",
            "Lower r* → some private investment recovers  (crowding-in effect)",
            "Net effect:  Y* ↓  but debt-to-GDP may improve if  μ < 1",
        ],
    ),
    dict(
        label     = "Tax cut  (t ↓ −3 pp)",
        kind      = "exp",
        channel   = "IS right + IS flattens",
        delta     = dict(t=-0.03),
        narrative = (
            "Tax cut → disposable income rises → α = 1−c(1−t) falls → "
            "IS flattens and shifts right. Fiscal multiplier μ = 1/α rises."
        ),
        steps = [
            "Tax rate t falls → disposable income  (1−t)·Y  rises for every household",
            "α = 1 − c(1−t) decreases → IS becomes flatter (more elastic to output)",
            "Multiplier  μ = 1/α  rises → larger output effect per unit of G",
            "IS shifts right: for every r, equilibrium Y is now higher",
            "Move along LM → new equilibrium: higher Y*, higher r* (crowding out)",
        ],
    ),
    
# COORDINATED POLICY 
    dict(
        label     = "Coordinated stimulus  (M↑ + G↑)",
        kind      = "mix",
        channel   = "IS right + LM right",
        delta     = dict(MP=+20, G=+2.5),
        narrative = (
            "Joint monetary + fiscal expansion: both LM (right) and IS (right) shift. "
            "Large ΔY with smaller Δr than either policy alone — the policy-mix sweet spot."
        ),
        steps = [
            "BCB expands money supply → LM shifts right → downward pressure on r",
            "Treasury raises G → IS shifts right → upward pressure on Y",
            "Combined:  large ΔY*  with small Δr*  (forces partially cancel each other)",
            "Useful when recession is deep and both demand and credit need support",
            "Risk: if economy is near capacity, may generate inflation instead of output",
        ],
    ),
    dict(
        label     = "Coordinated tightening  (M↓ + G↓)",
        kind      = "mix",
        channel   = "IS left + LM left",
        delta     = dict(MP=-20, G=-2),
        narrative = (
            "Dual contraction (IMF-style stabilisation). "
            "Both curves shift left → strong Y* reduction; "
            "Δr is ambiguous as the two opposing forces partially cancel."
        ),
        steps = [
            "BCB tightens money supply → LM shifts left → r rises",
            "Government cuts spending → IS shifts left → Y falls",
            "Combined:  large ΔY* reduction,  ambiguous Δr (forces partly offset)",
            "Classic IMF stabilisation package for overheating / BoP-crisis economies",
            "Intended: reduce inflation, improve current account, restore fiscal space",
        ],
    ),
]

KIND_COLOR = {"exp": PAL["EXP"], "con": PAL["CON"], "mix": PAL["MIX"]}
print(f"✓ {len(SIMULATIONS)} policy simulations defined")
for s in SIMULATIONS:
    p = copy_params(BL, **{k: BL[k] + v for k, v in s['delta'].items()
                           if k in BL})
    Y, r = equilibrium(p)
    print(f"  [{s['kind'].upper()}] {s['label'][:46]:<46} "
          f"Y*={Y:.1f} ({Y-BL_Y:+.1f})  r*={r:.2f}% ({r-BL_r:+.2f} pp)")

✓ 9 policy simulations defined
  [EXP] Money supply increase  (M/P ↑ +25)             Y*=150.6 (+15.9)  r*=-2.17% (-5.86 pp)
  [CON] Money supply contraction  (M/P ↓ −25)          Y*=118.8 (-15.9)  r*=9.56% (+5.86 pp)
  [EXP] Selic rate cut  (−300 bp)                      Y*=144.2 (+9.5)  r*=0.18% (-3.52 pp)
  [CON] Selic rate hike  (+300 bp)                     Y*=122.0 (-12.7)  r*=8.39% (+4.69 pp)
  [EXP] Gov. spending increase  (G ↑ +3 pp GDP)        Y*=137.6 (+2.9)  r*=4.78% (+1.09 pp)
  [CON] Fiscal austerity  (G ↓ −3 pp GDP)              Y*=131.9 (-2.9)  r*=2.61% (-1.09 pp)
  [EXP] Tax cut  (t ↓ −3 pp)                           Y*=137.5 (+2.8)  r*=4.77% (+1.08 pp)
  [MIX] Coordinated stimulus  (M↑ + G↑)                Y*=149.8 (+15.1)  r*=-0.09% (-3.78 pp)
  [MIX] Coordinated tightening  (M↓ + G↓)              Y*=120.1 (-14.6)  r*=7.66% (+3.97 pp)


In [ ]:
#Static overview of the model
fig_ov, axes_ov = plt.subplots(3, 3, figsize=(16, 13), facecolor=PAL["BG"])
fig_ov.suptitle(
    "Brazil IS-LM — Policy Simulation Overview  ·  9 scenarios  ·  2024 Baseline",
    fontsize=13, fontweight="bold", y=0.99
)

IS_BL_r = IS(Y_RANGE, BL)
LM_BL_r = LM(Y_RANGE, BL)

for ax, sim in zip(axes_ov.flat, SIMULATIONS):
    ax.set_facecolor(PAL["PANEL"])
    ax.grid(True, color=PAL["GRID"], lw=0.7, zorder=0)

    # baseline curves
    Xi0, Ri0 = clip_curve(Y_RANGE, IS_BL_r)
    Xl0, Rl0 = clip_curve(Y_RANGE, LM_BL_r)
    ax.plot(Xi0, Ri0, color=PAL["IS"], lw=1.8, alpha=0.50)
    ax.plot(Xl0, Rl0, color=PAL["LM"], lw=1.8, alpha=0.50)
    ax.scatter([BL_Y], [BL_r], color=PAL["EQ_BL"], s=65, zorder=5)

# shocked curves
    p2 = copy_params(BL, **{k: BL[k] + v for k, v in sim['delta'].items() if k in BL})
    Xi2, Ri2 = clip_curve(Y_RANGE, IS(Y_RANGE, p2))
    Xl2, Rl2 = clip_curve(Y_RANGE, LM(Y_RANGE, p2))
    ax.plot(Xi2, Ri2, color=PAL["IS2"], lw=2.0, ls="--")
    ax.plot(Xl2, Rl2, color=PAL["LM2"], lw=2.0, ls="--")

    Yn, Rn = equilibrium(p2)
    ax.scatter([Yn], [Rn], color=PAL["EQ_NEW"], s=80, marker="D", zorder=5)
    ax.annotate(
        "", xy=(Yn, Rn), xytext=(BL_Y, BL_r),
        arrowprops=dict(arrowstyle="-|>", color="#888", lw=1.2,
                        connectionstyle="arc3,rad=0.22")
    )

    dY = Yn - BL_Y;  dR = Rn - BL_r
    kc = KIND_COLOR[sim['kind']]
    ax.text(0.04, 0.97,
            f"ΔY = {dY:+.1f}\nΔr = {dR:+.2f} pp\n"
            f"Y*={Yn:.1f}   r*={Rn:.2f}%",
            transform=ax.transAxes, fontsize=8, va="top",
            family="monospace", color=kc,
            bbox=dict(boxstyle="round,pad=0.3", fc=PAL["BG"], ec="#ccc", alpha=0.9))
    ax.text(0.97, 0.03, sim['channel'],
            transform=ax.transAxes, fontsize=7.5, ha="right", va="bottom",
            color=kc, style="italic")
    short = sim['label'].split("(")[0].strip()
    ax.set_title(short, fontsize=8.5, pad=5, color=kc, fontweight="bold")
    ax.set_xlim(55, 145);  ax.set_ylim(*VIS)
    ax.set_xlabel("Y", fontsize=8);  ax.set_ylabel("r (%)", fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout(rect=[0, 0.025, 1, 0.98])
fig_ov.text(
    0.5, 0.005,
    "Solid = baseline  ·  Dashed = shocked  ·  ◆ = new equilibrium  ·  "
    "● = baseline  ·  Arrow = direction of shift  |  "
    "Green = expansionary   Red = contractionary   Blue = mixed",
    ha="center", fontsize=8, color="#888"
)
plt.savefig("brazil_islm_overview.png", dpi=150, bbox_inches="tight",
            facecolor=PAL["BG"])
plt.show()

In [ ]:
#Fully Interactive stimulation figure
# Figure skeleton 
FIG_W, FIG_H = 16, 14
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor=PAL["BG"])
fig.suptitle(
    "Brazil IS-LM — Interactive Policy Simulation  ·  2024 Baseline",
    fontsize=13, fontweight="bold", y=0.995, color="#1a1a18"
)

gs = gridspec.GridSpec(
    2, 2, figure=fig,
    hspace=0.38, wspace=0.28,
    left=0.06, right=0.97,
    top=0.970, bottom=0.385,
)

# IS-LM diagram (top-left) 
ax_main = fig.add_subplot(gs[0, 0])
ax_main.set_facecolor(PAL["PANEL"])
ax_main.grid(True, color=PAL["GRID"], lw=0.7, zorder=0)
ax_main.set_xlim(55, 145);  ax_main.set_ylim(*VIS)
ax_main.set_xlabel("Output index  Y", fontsize=11)
ax_main.set_ylabel("Real interest rate  r  (%)", fontsize=11)
ax_main.set_title("IS-LM Equilibrium  —  2024 baseline", fontsize=11, pad=7)

# draw baseline curves (never change)
Xi_bl, Ri_bl = clip_curve(Y_RANGE, IS(Y_RANGE, BL))
Xl_bl, Rl_bl = clip_curve(Y_RANGE, LM(Y_RANGE, BL))
ax_main.plot(Xi_bl, Ri_bl, color=PAL["IS"], lw=2.5, zorder=3)
ax_main.plot(Xl_bl, Rl_bl, color=PAL["LM"], lw=2.5, zorder=3)
pt_bl = ax_main.scatter([BL_Y], [BL_r], color=PAL["EQ_BL"], s=120, zorder=6)

# mutable artists (updated by callbacks)
line_is_sh, = ax_main.plot([], [], color=PAL["IS2"], lw=2.2, ls="--", zorder=4)
line_lm_sh, = ax_main.plot([], [], color=PAL["LM2"], lw=2.2, ls="--", zorder=4)
pt_new      = ax_main.scatter([], [], color=PAL["EQ_NEW"], s=130, marker="D", zorder=7)
arrow_patch = ax_main.annotate(
    "", xy=(BL_Y, BL_r), xytext=(BL_Y, BL_r),
    arrowprops=dict(arrowstyle="-|>", color="#999", lw=1.4,
                    connectionstyle="arc3,rad=0.2"),
    zorder=5,
)
arrow_patch.set_visible(False)
info_box = ax_main.text(
    0.03, 0.97, "", transform=ax_main.transAxes,
    fontsize=8.5, va="top", family="monospace", zorder=8,
    bbox=dict(boxstyle="round,pad=0.4", fc=PAL["BG"], ec="#ccc", alpha=0.93),
)
ax_main.legend(
    handles=[
        Line2D([0],[0], color=PAL["IS"],     lw=2.5, label="IS baseline"),
        Line2D([0],[0], color=PAL["LM"],     lw=2.5, label="LM baseline"),
        Line2D([0],[0], color=PAL["IS2"],    lw=2, ls="--", label="IS shifted"),
        Line2D([0],[0], color=PAL["LM2"],    lw=2, ls="--", label="LM shifted"),
        Line2D([0],[0], marker="o", color="w",
               markerfacecolor=PAL["EQ_BL"],  ms=8, label="Eq. baseline"),
        Line2D([0],[0], marker="D", color="w",
               markerfacecolor=PAL["EQ_NEW"], ms=8, label="Eq. new"),
    ],
    fontsize=8.5, framealpha=0.88, loc="upper right"
)

# Transmission narrative (top-right) 
ax_narr = fig.add_subplot(gs[0, 1])
ax_narr.set_facecolor(PAL["PANEL"])
ax_narr.axis("off")
ax_narr.set_title("Transmission Mechanism", fontsize=11, pad=7)

# Metrics table (bottom, full width)
ax_met = fig.add_subplot(gs[1, :])
ax_met.set_facecolor(PAL["PANEL"])
ax_met.axis("off")
ax_met.set_title("Equilibrium Comparison  —  Baseline vs Shocked", fontsize=10, pad=5)

# Fine-tune sliders 
SL_SPECS = [
    # (label,  param_key,  lo,    hi,   step, init,       display_scale, fmt)
    ("M/P  money supply",   "MP",  50,  160,   1.0, BL["MP"],  1.0,    "{:.0f}"),
    ("G  gov. spending (%GDP)", "G",  14,   32,  0.5, BL["G"],   1.0,    "{:.1f}"),
    ("t  tax rate (%)",        "t",  25,   45,  0.5, BL["t"]*100, 0.01, "{:.1f}%"),
    ("πᵉ  exp. inflation (%)", "pi",  2,   16,  0.1, BL["pi"],  1.0,    "{:.1f}%"),
]

slider_objs = []
for i, (lbl, key, lo, hi, step, init, scale, fmt) in enumerate(SL_SPECS):
    ax_sl = plt.axes(
        [0.06, 0.340 - i * 0.042, 0.60, 0.025],
        facecolor="#F0EDE6"
    )
    sl = Slider(ax_sl, lbl, lo, hi, valinit=init, valstep=step,
                color=PAL["IS"], track_color="#DDD")
    sl.label.set_fontsize(8.5)
    sl.valtext.set_fontsize(8.5)
    slider_objs.append((sl, key, scale))

# Simulation buttons 
RESET_SIM = dict(
    label="── Reset to 2024 baseline ──",
    kind="rst", channel="—", delta={},
    narrative="All parameters reset to the 2024 Brazil baseline.",
    steps=[],
)
ALL_SIMS = SIMULATIONS + [RESET_SIM]

BTN_COLS = 5                          # buttons per row
BTN_W    = 0.173
BTN_H    = 0.045
BTN_GAP  = 0.010
BTN_LEFT = 0.06
BTN_TOP  = 0.295

button_objs = []
for idx, sim in enumerate(ALL_SIMS):
    row = idx // BTN_COLS
    col = idx %  BTN_COLS
    x   = BTN_LEFT + col * (BTN_W + BTN_GAP)
    y   = BTN_TOP  - row * (BTN_H  + BTN_GAP + 0.002)
    kc  = KIND_COLOR.get(sim['kind'], "#888888")
    fc  = {"exp": "#E1F5EE", "con": "#FAECE7",
           "mix": "#E6F1FB", "rst": "#F1EFE8"}.get(sim['kind'], "#F0EDE6")
    ax_b = plt.axes([x, y, BTN_W, BTN_H], facecolor=fc)
    btn  = Button(ax_b, sim['label'].split("(")[0].strip(),
                  color=fc, hovercolor="#D5D0C8")
    btn.label.set_fontsize(7.8)
    btn.label.set_color(kc)
    button_objs.append(btn)

#  HELPER: redraw transmission narrative
def draw_narrative(sim):
    ax_narr.cla()
    ax_narr.set_facecolor(PAL["PANEL"])
    ax_narr.axis("off")
    ax_narr.set_title("Transmission Mechanism", fontsize=11, pad=7)

    if not sim or not sim['steps']:
        ax_narr.text(0.05, 0.90,
                     "2024 baseline restored.\nAll curves reset.",
                     transform=ax_narr.transAxes,
                     fontsize=9, va="top", color="#555")
        return

    kc = KIND_COLOR.get(sim['kind'], "#888")

# title
    ax_narr.text(0.03, 0.98, sim['label'],
                 transform=ax_narr.transAxes, fontsize=9, va="top",
                 fontweight="bold", color=kc)
# channel badge
    ax_narr.text(0.03, 0.90,
                 f"Channel:  {sim['channel']}",
                 transform=ax_narr.transAxes, fontsize=8.5, va="top",
                 family="monospace", color=kc)
# narrative sentence (word-wrap at ~55 chars)
    narr = sim['narrative']
    wrapped = "\n".join(
        narr[i:i+55] for i in range(0, len(narr), 55)
    )
    ax_narr.text(0.03, 0.82, wrapped,
                 transform=ax_narr.transAxes, fontsize=8, va="top",
                 color="#555", style="italic", linespacing=1.5)

# numbered steps
    Y_START = 0.66
    STEP_H  = 0.118
    for i, step in enumerate(sim['steps']):
        yp = Y_START - i * STEP_H
# circle bullet
        circ = plt.Circle(
            (0.055, yp), 0.030,
            transform=ax_narr.transAxes,
            color=kc, alpha=0.18, clip_on=False
        )
        ax_narr.add_patch(circ)
        ax_narr.text(0.055, yp, str(i + 1),
                     transform=ax_narr.transAxes, fontsize=8,
                     ha="center", va="center",
                     color=kc, fontweight="bold")
# step text (wrap at 50 chars)
        lines = [step[j:j+50] for j in range(0, len(step), 50)]
        ax_narr.text(0.115, yp, "\n".join(lines),
                     transform=ax_narr.transAxes, fontsize=8,
                     va="center", color="#333", linespacing=1.4)

#  HELPER: redraw metrics comparison table
def draw_metrics(p_cur, Y_cur, r_cur):
    ax_met.cla()
    ax_met.set_facecolor(PAL["PANEL"])
    ax_met.axis("off")
    ax_met.set_title("Equilibrium Comparison  —  Baseline vs Shocked",
                     fontsize=10, pad=5)

    i_cur  = nominal_rate(p_cur, r_cur)
    mu_cur = multiplier(p_cur)
    dY     = Y_cur - BL_Y
    dr     = r_cur - BL_r
    di     = i_cur - BL_i
    dmu    = mu_cur - BL_mu

    def s(v): return "+" if v >= 0 else ""

# (metric label, baseline val, current val, delta str, interpretation)
    rows = [
        ("Y*  output index",
         f"{BL_Y:.2f}", f"{Y_cur:.2f}", f"{s(dY)}{dY:.2f}",
         "Output " + ("expanded" if dY > 0.01 else ("contracted" if dY < -0.01 else "unchanged"))),
        ("r*  real interest rate",
         f"{BL_r:.2f}%", f"{r_cur:.2f}%", f"{s(dr)}{dr:.2f} pp",
         "Rate " + ("rose → investment ↓" if dr > 0.01 else
                    ("fell → investment ↑" if dr < -0.01 else "unchanged"))),
        ("i*  nominal rate (Fisher)",
         f"{BL_i:.2f}%", f"{i_cur:.2f}%", f"{s(di)}{di:.2f} pp",
         "i* = r* + πᵉ"),
        ("μ   fiscal multiplier",
         f"{BL_mu:.4f}×", f"{mu_cur:.4f}×", f"{s(dmu)}{dmu:.4f}",
         "μ = 1 / (1 − c(1−t))"),
        ("G   gov. spending",
         f"{BL['G']:.1f}%", f"{p_cur['G']:.1f}%",
         f"{s(p_cur['G']-BL['G'])}{p_cur['G']-BL['G']:.1f} pp",
         "Fiscal stance"),
        ("M/P real money supply",
         f"{BL['MP']:.0f}", f"{p_cur['MP']:.0f}",
         f"{s(p_cur['MP']-BL['MP'])}{p_cur['MP']-BL['MP']:.0f}",
         "Monetary stance"),
        ("t   tax rate",
         f"{BL['t']*100:.1f}%", f"{p_cur['t']*100:.1f}%",
         f"{s(p_cur['t']-BL['t'])}{(p_cur['t']-BL['t'])*100:.1f} pp",
         "Tax wedge"),
        ("πᵉ  exp. inflation",
         f"{BL['pi']:.2f}%", f"{p_cur['pi']:.2f}%",
         f"{s(p_cur['pi']-BL['pi'])}{p_cur['pi']-BL['pi']:.2f} pp",
         "Fisher: i = r + πᵉ"),
    ]

# column proportions
    CW   = [2.0, 1.0, 1.0, 1.1, 2.2]
    CW_S = sum(CW)
    COL_HEADERS = ["Metric", "Baseline", "Shocked", "Δ", "Interpretation"]
    RH   = 0.100
    Y0   = 0.95

    def cx(col_idx):
        return 0.01 + sum(CW[:col_idx]) / CW_S * 0.97

    # header row
    for ci, hdr in enumerate(COL_HEADERS):
        ax_met.text(cx(ci), Y0, hdr,
                    transform=ax_met.transAxes, fontsize=8,
                    fontweight="bold", color="#1a1a18", va="top")

    for ri, row in enumerate(rows):
        yp = Y0 - (ri + 1) * RH
        bg = PAL["PANEL"] if ri % 2 == 0 else PAL["BG"]
        ax_met.add_patch(plt.Rectangle(
            (0, yp - 0.01), 1.0, RH - 0.01,
            fc=bg, ec=PAL["GRID"], lw=0.4,
            transform=ax_met.transAxes, clip_on=False
        ))
        for ci, cell in enumerate(row):
            color = "#1a1a18"
            if ci == 3:               # Δ column: colour by direction
                try:
                    raw = cell.replace("+","").replace("pp","").replace(
                          "×","").replace("%","").strip()
                    v = float(raw)
                    if ri == 0:       # ΔY: positive = good
                        color = PAL["EXP"] if v > 0.005 else (
                                PAL["CON"] if v < -0.005 else "#888")
                    elif ri == 1:     # Δr: negative = good (lower rates)
                        color = PAL["EXP"] if v < -0.005 else (
                                PAL["CON"] if v > 0.005 else "#888")
                    elif ri == 2:     # Δi: same as Δr
                        color = PAL["EXP"] if v < -0.005 else (
                                PAL["CON"] if v > 0.005 else "#888")
                except Exception:
                    pass
            ax_met.text(cx(ci), yp + RH * 0.38, cell,
                        transform=ax_met.transAxes, fontsize=8,
                        va="center", color=color,
                        family="monospace" if ci in (1, 2, 3) else "sans-serif")

#  HELPER: apply a parameter dict and redraw everything
_current_p = dict(BL)   # mutable state shared across callbacks

def apply_and_redraw(p, sim=None):
    global _current_p
    _current_p = dict(p)

    Y_new, r_new = equilibrium(p)
    dY = Y_new - BL_Y;   dr = r_new - BL_r
    s  = lambda v: "+" if v >= 0 else ""

    # IS / LM shifted curves
    r_is2 = IS(Y_RANGE, p);   r_lm2 = LM(Y_RANGE, p)
    IS_changed = np.any(np.abs(r_is2 - IS(Y_RANGE, BL)) > 0.01)
    LM_changed = np.any(np.abs(r_lm2 - LM(Y_RANGE, BL)) > 0.01)

    if IS_changed:
        Xi2, Ri2 = clip_curve(Y_RANGE, r_is2)
        line_is_sh.set_data(Xi2, Ri2)
    else:
        line_is_sh.set_data([], [])

    if LM_changed:
        Xl2, Rl2 = clip_curve(Y_RANGE, r_lm2)
        line_lm_sh.set_data(Xl2, Rl2)
    else:
        line_lm_sh.set_data([], [])

    if IS_changed or LM_changed:
        pt_new.set_offsets([[Y_new, r_new]])
        arrow_patch.set_visible(True)
        arrow_patch.xy     = (Y_new, r_new)
        arrow_patch.xyann  = (BL_Y, BL_r)
        # update arrowprops xy / xytext through set_position trick
        arrow_patch.set_position((BL_Y, BL_r))
        # rebuild annotation (matplotlib quirk)
        arrow_patch.xy = (Y_new, r_new)
    else:
        pt_new.set_offsets(np.empty((0, 2)))
        arrow_patch.set_visible(False)

    kc = KIND_COLOR.get(sim['kind'], "#888") if sim else "#888"
    title = (f"IS-LM — {sim['label'].split('(')[0].strip()}"
             if sim and sim['kind'] != 'rst' else
             "IS-LM Equilibrium  —  2024 baseline")
    ax_main.set_title(title, fontsize=10, pad=7, color=kc)

    info_box.set_text(
        f"Adjusted equilibrium\n"
        f"  Y* = {Y_new:.2f}  (Δ {s(dY)}{dY:.2f})\n"
        f"  r* = {r_new:.2f}%  (Δ {s(dr)}{dr:.2f} pp)\n"
        f"  i* = {nominal_rate(p, r_new):.2f}%  (Fisher)\n"
        f"  μ  = {multiplier(p):.4f}×\n"
        f"\nBaseline: Y={BL_Y:.2f}, r={BL_r:.2f}%"
    )

# sync sliders to current params (without re-triggering callbacks)
    for (sl, key, scale) in slider_objs:
        new_val = p[key] / scale
        if abs(sl.val - new_val) > 1e-9:
            sl.eventson = False
            sl.set_val(new_val)
            sl.eventson = True

    draw_narrative(sim)
    draw_metrics(p, Y_new, r_new)
    fig.canvas.draw_idle()

# Button callbacks 
def make_btn_handler(sim):
    def handler(event):
        p = copy_params(BL, **{k: BL[k] + v
                                for k, v in sim['delta'].items() if k in BL})
        apply_and_redraw(p, sim)
    return handler

for btn, sim in zip(button_objs, ALL_SIMS):
    btn.on_clicked(make_btn_handler(sim))

# Slider callbacks 
def make_slider_handler(key, scale):
    def handler(val):
        p = dict(_current_p)
        p[key] = val * scale
        apply_and_redraw(p, sim=None)
    return handler

for (sl, key, scale) in slider_objs:
    sl.on_changed(make_slider_handler(key, scale))

# Initial render 
apply_and_redraw(dict(BL), sim=SIMULATIONS[0])

plt.savefig("brazil_islm_simulation.png", dpi=140,
            bbox_inches="tight", facecolor=PAL["BG"])
plt.show()

print("\n✓ Interactive simulation figure ready")
print("  BUTTONS  — click any of the 9 policy scenarios to fire a simulation")
print("  SLIDERS  — fine-tune M/P, G, t, πᵉ live after selecting a scenario")
print("  LEFT     — IS-LM diagram with animated curve shift and equilibrium arrow")
print("  RIGHT    — 5-step transmission mechanism narrative")
print("  BOTTOM   — full metrics comparison table (baseline vs shocked)")


In [ ]:
# CONSOLE SUMMARY TABLE   

print("\n" + "═" * 90)
print("  BRAZIL IS-LM — POLICY SIMULATION SUMMARY")
print("═" * 90)
print(f"  {'Scenario':<45} {'Y*':>6}  {'r*':>7}  {'i*':>7}  {'ΔY':>8}  {'Δr':>9}  {'Kind'}")
print("─" * 90)
for sim in SIMULATIONS:
    p      = copy_params(BL, **{k: BL[k] + v for k, v in sim['delta'].items() if k in BL})
    Y_s, r_s = equilibrium(p)
    i_s    = nominal_rate(p, r_s)
    dY     = Y_s - BL_Y;   dr = r_s - BL_r
    kind_s = {"exp": "↑ expand", "con": "↓ contract", "mix": "↔ mixed"}[sim['kind']]
    print(f"  {sim['label'][:45]:<45} {Y_s:>6.2f}  {r_s:>6.2f}%  {i_s:>6.2f}%  "
          f"{dY:>+8.2f}  {dr:>+8.2f} pp  {kind_s}")
print("─" * 90)
print(f"  {'BASELINE (2024)':<45} {BL_Y:>6.2f}  {BL_r:>6.2f}%  {BL_i:>6.2f}%  "
      f"{'—':>8}  {'—':>9}")
print("═" * 90)